# There is a class called `GridSearch` which finds the best value for the imputation. See the video of `GridSearch` in `CampusX's` channel for learning. Here, we're directly moving too the code section.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("00 titanic.csv")

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Dropping un-necessary columns.

df.drop(columns = ['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace = True)

In [5]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [6]:
df.isnull().sum()

# Missing values are in columns 'Age' and 'Embarked'

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [7]:
# TRain/test split

x = df.drop('Survived', axis = 'columns')
y = df.drop(x, axis = 'columns')

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2)

In [9]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
338,3,male,45.00,0,0,8.05,S
21,2,male,34.00,0,0,13.00,S
70,2,male,32.00,0,0,10.50,S
78,2,male,0.83,0,2,29.00,S
347,3,female,NaN,1,0,16.10,S


In [24]:
numeric_features = ['Age', 'Fare']

numeric_data = [
    ('imputer', SimpleImputer(strategy = 'median')),
]

numeric_transformer = Pipeline(numeric_data)

In [33]:
categorical_features = ['Sex', 'Embarked']


categorical_data = [
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore')),
]


categorical_transformer = Pipeline(categorical_data)

In [34]:
# Define the preprocessor with ColumnTransformer
trans_ct = [
    ('num_transformer', numeric_transformer, numeric_features),
    ('cat_transformer', categorical_transformer, categorical_features),
]


preprocessor = ColumnTransformer(transformers=trans_ct, remainder='passthrough')

In [46]:
# Define the full pipeline with preprocessor and classifier
'''
data_pipeline = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
]

pipe = Pipeline(data_pipeline)
'''

data_pipeline = [
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),  # Add scaler here
    ('classifier', LogisticRegression())
]

pipe = Pipeline(data_pipeline)

In [47]:
from sklearn import set_config

set_config(display='diagram')


In [48]:
pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num_transformer',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Fare']),
                                                 ('cat_transformer',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked'])])),
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression())])

In [49]:
# Define the parameter grid for GridSearchCV
param_grid = {
    'preprocessor__num_transformer__imputer__strategy': ['mean', 'median'],
    'preprocessor__cat_transformer__imputer__strategy': ['most_frequent', 'constant'],
    'classifier__C': [0.1, 1.0, 10, 100]
}

# Perform GridSearchCV
grid_search = GridSearchCV(pipe, param_grid, cv=10)

In [50]:
grid_search.fit(x_train, y_train)

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1143: DataC

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('num_transformer',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         ['Age',
                                                                          'Fare']),
                                                                        ('cat_transformer',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('ohe',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Sex',
                                                                          'Embarked'])])),
                                       ('scaler', StandardScaler()),
                                       ('classifier', LogisticRegression())]),
             param_grid={'classifier__C': [0.1, 1.0, 10, 100],
                         'preprocessor__cat_transformer__imputer__strategy': ['most_frequent',
                                                                              'constant'],
                         'preprocessor__num_transformer__imputer__strategy': ['mean',
                                                                              'median']})

In [51]:
print(f"Best params:")
print(grid_search.best_params_)

Best params:
{'classifier__C': 0.1, 'preprocessor__cat_transformer__imputer__strategy': 'most_frequent', 'preprocessor__num_transformer__imputer__strategy': 'mean'}


In [52]:
print(f"Internal CV score: {grid_search.best_score_:.3f}")


Internal CV score: 0.808


In [53]:
import pandas as pd

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values("mean_test_score", ascending=False)
cv_results[['param_classifier__C', 'param_preprocessor__cat_transformer__imputer__strategy', 'param_preprocessor__num_transformer__imputer__strategy', 'mean_test_score']]


,param_classifier__C,param_preprocessor__cat_transformer__imputer__strategy,param_preprocessor__num_transformer__imputer__strategy,mean_test_score
0,0.1,most_frequent,mean,0.807746
1,0.1,most_frequent,median,0.807746
2,0.1,constant,mean,0.807746
3,0.1,constant,median,0.807746
4,1.0,most_frequent,mean,0.804969
6,1.0,constant,mean,0.804969
8,10,most_frequent,mean,0.804969
10,10,constant,mean,0.804969
5,1.0,most_frequent,median,0.803560
7,1.0,constant,median,0.803560


# I didn't understood the working of this code. I mean how Pipeline() and ColumnnTransformer() is working hand-in-hand.
# Moreover, learn the concept of `GridSearch() properly.

# Next: Day 39 - KNNImputer()